# TAL-64 — Deduplicazione jCityGov: quale chiave di identità è sicura?

**Obiettivo:** capire, su dati reali, come riconoscere "stesso atto trovato da due fonti" (Albo Pretorio + Amministrazione Trasparente) senza fondere per errore due atti realmente distinti.

Dati: `atti_trasparenza_staging` (1048 atti reali di Amministrazione Trasparente jCityGov, 7 comuni: Ragusa, Modica, Acireale, Scicli, Giarre, Sant'Agata li Battiati, Gravina di Catania — raccolti con `scripts/tal64_staging_trasparenza.py`) confrontati con `atti` reale (Albo Pretorio, `fonte_scraper='jcitygov'`).

**Perché questo notebook**: un primo giro veloce da riga di comando aveva concluso "CIG e (tipo,numero) sono sicuri, 0 falsi positivi" — ma quella verifica controllava solo *se* un match esisteva, non se fosse *quello giusto* quando ce n'erano più di uno. Qui verifichiamo la precisione, non solo la presenza.

In [1]:
import sqlite3

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

conn = sqlite3.connect("../talia.db")

staging = pd.read_sql_query(
    "SELECT * FROM atti_trasparenza_staging WHERE fonte_scraper='jcitygov_trasparenza'", conn
)
atti = pd.read_sql_query("SELECT * FROM atti WHERE fonte_scraper='jcitygov'", conn)

print(f"staging (Amministrazione Trasparente): {len(staging)} righe")
print(f"atti (Albo Pretorio, jcitygov, tutti i comuni): {len(atti)} righe")
print(f"comuni in staging: {staging['ente_id'].nunique()}")

staging (Amministrazione Trasparente): 1048 righe
atti (Albo Pretorio, jcitygov, tutti i comuni): 85435 righe
comuni in staging: 7


## Prova 1 — copertura dei campi candidati come chiave di identità

In [2]:
for campo in ("cig", "numero", "data_atto", "oggetto"):
    n = staging[campo].notna().sum()
    print(f"{campo:12s}: {n}/{len(staging)} ({n / len(staging):.0%})")

print()
print("numero='0' (sospetta sentinella):", (staging["numero"] == "0").sum())

cig         : 342/1048 (33%)
numero      : 749/1048 (71%)
data_atto   : 0/1048 (0%)
oggetto     : 1048/1048 (100%)

numero='0' (sospetta sentinella): 135


`data_atto` non è mai valorizzato: la chiave di fallback `(tipo, numero, data_atto)` ipotizzata nella spec originale di TAL-64 va quindi rivista — non è utilizzabile così com'è.

## Prova 2 — il CIG identifica un atto, o una gara (con più atti)?

Dom aveva confermato che un CIG identifica una gara in modo univoco. Ma una gara produce *più* atti nel tempo (determina a contrarre, aggiudicazione, liquidazione...). Se lo stesso CIG compare su più righe di `atti` con contenuto diverso, usare il solo CIG come chiave di deduplicazione fonderebbe atti realmente distinti.

In [3]:
cig_atti = atti[atti["cig"].notna()]
per_cig = cig_atti.groupby(["ente_id", "cig"]).size().sort_values(ascending=False)

print("CIG con più di un atto nello stesso ente (Albo Pretorio):")
print(f"  {(per_cig > 1).sum()} / {len(per_cig)} CIG distinti")
print()
print("Distribuzione (quanti atti per CIG):")
print(per_cig.value_counts().sort_index())

CIG con più di un atto nello stesso ente (Albo Pretorio):
  5720 / 14362 CIG distinti

Distribuzione (quanti atti per CIG):
1     8642
2     3495
3      968
4      460
5      192
6      136
7      105
8       60
9       63
10      38
11      27
12      31
13      19
14      15
15      15
16      18
17       5
18       8
19       4
20       4
21       1
22       4
23       2
24       3
25       6
27       4
28       2
29       1
30       3
31       6
32       3
34       2
35       1
36       1
38       3
39       2
40       2
41       1
44       1
48       1
55       1
60       1
63       1
64       1
69       1
70       1
85       1
91       1
Name: count, dtype: int64


In [4]:
# Esempio concreto: un CIG con più atti, per vedere se sono davvero documenti diversi.
# Scartiamo candidati che non hanno la forma di un CIG vero (10 caratteri alfanumerici,
# con almeno una cifra: "ORIGINARIO" è 10 lettere ma non è un CIG, è un placeholder
# finito nel campo per un bug di estrazione — vedi nota più sotto).
cig_index = per_cig.index.get_level_values("cig")
forma_valida = cig_index.str.match(r"^[A-Z0-9]{10}$") & cig_index.str.contains(r"\d")
cig_veri = per_cig[forma_valida]
esempio_cig = cig_veri[cig_veri > 3].index[0]
ente_id, cig = esempio_cig

esempio = atti[(atti["ente_id"] == ente_id) & (atti["cig"] == cig)][
    ["id", "tipo", "numero", "oggetto", "url_fonte"]
]
print(f"CIG {cig!r}, ente_id {ente_id}: {len(esempio)} atti diversi")
esempio

CIG 'A0296F795D', ente_id 24: 85 atti diversi


,id,tipo,numero,oggetto,url_fonte
9383,13468,atti amministrativi /determina dirigenziale,935,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
9411,13496,atti amministrativi /determina dirigenziale,884,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
9498,13583,atti amministrativi /determina dirigenziale,871,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
39359,45938,atti amministrativi /determina dirigenziale,711,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
39390,45969,atti amministrativi /determina dirigenziale,658,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
...,...,...,...,...,...
77769,100336,atti amministrativi /determina dirigenziale,1155,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
78752,106085,atti amministrativi /determina dirigenziale,1321,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
78802,106135,atti amministrativi /determina dirigenziale,1321,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...
78846,106179,atti amministrativi /determina dirigenziale,1253,PAL 2021 – QSFP- EDUCATIVA DOMICILIARE- AZIONE 2- LIQUIDAZIONE ALLA COOPERAT...,https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/p...


**Conclusione Prova 2:** confermato — lo stesso CIG compare legittimamente su più atti distinti (tipo/numero/oggetto diversi: sono le tappe della stessa gara, non copie dello stesso documento). **Il CIG da solo non è una chiave di identità sicura per l'atto** — lo è per la *gara*, non per il singolo documento. Il primo giro da riga di comando aveva un bug metodologico: prendeva il primo risultato (`fetchone()`) senza controllare se ce ne fossero altri con lo stesso CIG.

## Prova 3 — chiavi più strette: `(cig, numero)` e `(cig, numero, tipo)`

In [5]:
for chiave in (("ente_id", "cig", "numero"), ("ente_id", "cig", "numero", "tipo")):
    sotto = cig_atti[cig_atti["numero"].notna() & (cig_atti["numero"] != "0")]
    conteggi = sotto.groupby(list(chiave)).size()
    ambigui = (conteggi > 1).sum()
    print(f"{chiave}: {ambigui}/{len(conteggi)} gruppi ambigui (>1 atto)")

('ente_id', 'cig', 'numero'): 1523/24438 gruppi ambigui (>1 atto)
('ente_id', 'cig', 'numero', 'tipo'): 1314/24687 gruppi ambigui (>1 atto)


`(ente_id, cig, numero)` è ambiguo (>1 atto) nel 6,2% dei gruppi (1523/24438) — non "quasi sempre univoco" come ipotizzato prima di eseguire il notebook, un errore c'è ancora abbastanza spesso da non poterlo ignorare. Aggiungere `tipo` migliora poco (1314/24687). Nella cella sopra è anche emerso un problema di qualità dati collaterale: uno dei CIG più "ripetuti" non è un CIG reale ma la stringa letterale `"ORIGINARIO"` (91 atti) — un placeholder finito nel campo `cig`, non un codice di gara. Andrebbe ripulito a monte (probabilmente in `estrai_cig()` o nella sorgente), ma non blocca questa analisi: il fenomeno "un CIG vero copre più atti" resta confermato anche togliendo questo caso.

## Prova 4 — con la chiave corretta, quanti atti di staging sono davvero già noti?

In [6]:
chiave_atti = atti.dropna(subset=["cig", "numero"])
chiave_atti = chiave_atti[chiave_atti["numero"] != "0"]
conteggi = chiave_atti.groupby(["ente_id", "cig", "numero"]).size()
chiavi_univoche = conteggi[conteggi == 1].index
indice = (
    chiave_atti.set_index(["ente_id", "cig", "numero"])
    .loc[chiavi_univoche, "url_fonte"]
    .rename("url_atti")
)

staging_con_chiave = staging.dropna(subset=["cig", "numero"])
staging_con_chiave = staging_con_chiave[staging_con_chiave["numero"] != "0"]

confronto = staging_con_chiave.merge(
    indice, left_on=["ente_id", "cig", "numero"], right_index=True, how="inner"
)

print(f"Staging con CIG+numero validi: {len(staging_con_chiave)}/{len(staging)}")
print(f"Match univoco (ente,cig,numero) contro Albo Pretorio: {len(confronto)}")

stesso_url = (confronto["url_fonte"] == confronto["url_atti"]).sum()
diverso = confronto[confronto["url_fonte"] != confronto["url_atti"]]

print(f"  di cui stesso url_fonte (dedup già gratuita via UNIQUE): {stesso_url}")
print(f"  di cui url_fonte DIVERSO (il caso vero da deduplicare): {len(diverso)}")

Staging con CIG+numero validi: 159/1048
Match univoco (ente,cig,numero) contro Albo Pretorio: 149
  di cui stesso url_fonte (dedup già gratuita via UNIQUE): 148
  di cui url_fonte DIVERSO (il caso vero da deduplicare): 1


In [7]:
print("Esempi di url_fonte diverso, stesso (ente,cig,numero):")
for _, riga in diverso.head(5).iterrows():
    print(" ", (riga["ente_id"], riga["cig"], riga["numero"]))
    print("   staging:", riga["url_fonte"])
    print("   atti   :", riga["url_atti"])

Esempi di url_fonte diverso, stesso (ente,cig,numero):
  (24, 'B432E31727', '1969')
   staging: https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g-tag/-/papca/display/4097636?p_p_state=pop_up
   atti   : https://giarre.trasparenza-valutazione-merito.it/web/trasparenza/papca-g/-/papca/display/3926559?p_p_state=pop_up


## Conclusione (numeri reali dall'esecuzione)

1. **`data_atto` va tolto dalla chiave di fallback** — 0/1048, nessuna eccezione.
2. **Il CIG da solo non basta**: 5720/14362 CIG (40%) coprono più di un atto nell'Albo Pretorio — confermato anche al netto del CIG-placeholder `"ORIGINARIO"` trovato per caso (91 atti, bug di estrazione a parte).
3. **`(ente_id, cig, numero)` è ambiguo nel 6,2% dei casi** (1523/24438) — meglio del CIG da solo, ma non perfetto: un'implementazione reale deve comunque gestire l'ambiguità residua (es. non deduplicare quando la chiave è ambigua, inserire comunque come atto nuovo), non assumerla risolta.
4. **Il caso interessante esiste ma è raro in questo campione**: solo 159/1048 (15%) atti di Amministrazione Trasparente hanno sia CIG sia numero utilizzabili; di questi, 149 trovano un match univoco nell'Albo Pretorio, **148 con lo stesso identico `url_fonte`** (dedup già gratuita) e **1 solo con `url_fonte` diverso** — il vero caso da deduplicare. Non è un'ipotesi: esiste, ma su jCityGov è raro, non il caso comune.
5. **L'85% degli atti di Amministrazione Trasparente non ha CIG+numero utilizzabili** (soprattutto bandi di concorso, che non hanno CIG per natura) — per quella fetta grande la deduplicazione via CIG non si applica comunque; serve il fallback `(tipo, numero)` già provato nel Tentativo 1, con la stessa cautela sulla sentinella `numero="0"`.

**Correzione rispetto a TAL-64 Tentativo 1**: la card riportava "CIG/numero+tipo: 0 falsi positivi" — quello era vero solo perché il primo controllo prendeva un solo risultato a caso (`fetchone()`) senza verificare se il CIG avesse altri atti candidati. Con l'analisi completa qui sopra, il CIG da solo avrebbe prodotto falsi positivi nel 40% dei casi; la chiave `(cig, numero)` regge molto meglio ma non è perfetta (6,2% di ambiguità residua da gestire esplicitamente in `inserisci_atto()`, non da ignorare).